# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# Preprocesador para árboles: solo imputación por mediana
tree_preprocessor = ColumnTransformer(
    [("num", SimpleImputer(strategy="median"), feature_columns)]
)

# Preprocesador para modelos sensibles a la escala: imputación + estandarización
scaled_preprocessor = ColumnTransformer(
    [("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), feature_columns)]
)

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

Si el scaler se ajusta con todo `X_train` antes de la CV, la media y la desviación estándar incluyen las filas que luego actúan como validación en cada fold. Eso es fuga de información (data leakage).

## 3. Modelos candidatos

In [5]:
models = {
    "DecisionTree": Pipeline([
        ("preprocess", clone(tree_preprocessor)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "LogisticRegression": Pipeline([
        ("preprocess", clone(scaled_preprocessor)),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "KNN": Pipeline([
        ("preprocess", clone(scaled_preprocessor)),
        ("model", KNeighborsClassifier()),
    ]),
}

## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipeline in models.items():
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring="accuracy", return_train_score=True)
    rows.append({
        "model": name,
        "train_accuracy_mean": scores["train_score"].mean(),
        "val_accuracy_mean": scores["test_score"].mean(),
        "val_accuracy_std": scores["test_score"].std(),
    })

initial_comparison = pd.DataFrame(rows).set_index("model").sort_values("val_accuracy_mean", ascending=False)
initial_comparison.round(4)

,train_accuracy_mean,val_accuracy_mean,val_accuracy_std
model,,,
LogisticRegression,1.0000,0.9791,0.0277
KNN,0.9771,0.9507,0.0353
DecisionTree,1.0000,0.9022,0.0586


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "DecisionTree": {
        "model__max_depth": [2, 3, 4, 5, None],
        "model__min_samples_leaf": [1, 3, 5],
    },
    "LogisticRegression": {
        "model__C": [0.01, 0.1, 1, 10],
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
    },
}

searches = {}
rows = []
for name, pipeline in models.items():
    search = GridSearchCV(
        pipeline, param_grids[name], scoring="accuracy", cv=cv, return_train_score=True
    )
    search.fit(X_train, y_train)
    searches[name] = search
    best = search.best_index_
    rows.append({
        "model": name,
        "best_params": search.best_params_,
        "cv_accuracy": search.best_score_,
        "cv_std": search.cv_results_["std_test_score"][best],
        "train_accuracy": search.cv_results_["mean_train_score"][best],
    })

tuned_comparison = pd.DataFrame(rows).set_index("model").sort_values("cv_accuracy", ascending=False)
tuned_comparison

,best_params,cv_accuracy,cv_std,train_accuracy
model,,,,
LogisticRegression,{'model__C': 1},0.979064,0.027713,1.000000
KNN,"{'model__n_neighbors': 7, 'model__weights': 'u...",0.957882,0.040520,0.964773
DecisionTree,"{'model__max_depth': 4, 'model__min_samples_le...",0.902217,0.058617,0.998246


## 6. Selección y persistencia del campeón

In [8]:
import sklearn

champion_name = max(searches, key=lambda name: searches[name].best_score_)
champion = searches[champion_name].best_estimator_
print("Campeón:", champion_name, searches[champion_name].best_params_)

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

best = searches[champion_name].best_index_
training_metadata = {
    "champion": champion_name,
    "best_params": searches[champion_name].best_params_,
    "cv_accuracy": float(searches[champion_name].best_score_),
    "cv_std": float(searches[champion_name].cv_results_["std_test_score"][best]),
    "cv_folds": cv.get_n_splits(),
    "scoring": "accuracy",
    "random_state": RANDOM_STATE,
    "features": feature_columns,
    "target": contract["target"],
    "n_train": len(train_df),
    "candidates": {name: float(s.best_score_) for name, s in searches.items()},
    "sklearn_version": sklearn.__version__,
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
)

cv_results = pd.concat(
    [pd.DataFrame(s.cv_results_).assign(model=name) for name, s in searches.items()],
    ignore_index=True,
)
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)
print(json.dumps(training_metadata, indent=2, ensure_ascii=False, default=str))

Campeón: LogisticRegression {'model__C': 1}
{
  "champion": "LogisticRegression",
  "best_params": {
    "model__C": 1
  },
  "cv_accuracy": 0.9790640394088669,
  "cv_std": 0.027713464392187602,
  "cv_folds": 5,
  "scoring": "accuracy",
  "random_state": 42,
  "features": [
    "alcohol",
    "malic_acid",
    "ash",
    "alcalinity_of_ash",
    "magnesium",
    "total_phenols",
    "flavanoids",
    "nonflavanoid_phenols",
    "proanthocyanins",
    "color_intensity",
    "hue",
    "od280/od315_of_diluted_wines",
    "proline"
  ],
  "target": "target",
  "n_train": 142,
  "candidates": {
    "DecisionTree": 0.9022167487684729,
    "LogisticRegression": 0.9790640394088669,
    "KNN": 0.9578817733990148
  },
  "sklearn_version": "1.8.0"
}


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

El campeón es **LogisticRegression (C = 1)**, con accuracy CV = **0.979 ± 0.028**.

- **Desempeño:** es el mejor de los tres (KNN 0.958 ± 0.041 con k = 7, DecisionTree 0.902 ± 0.059 con max_depth = 4).
- **Variabilidad:** tiene la menor desviación estándar entre folds, así que su rendimiento es el más estable.
- **Sobreajuste:** su accuracy de entrenamiento es 1.0, pero la brecha con validación es pequeña (aprox. 2 pp). En cambio, el árbol también llega a aprox. 1.0 en train y cae a 0.90 en validación, un sobreajuste claro.
- **Complejidad:** es un modelo lineal con 13 × 3 coeficientes, rápido de entrenar y de predecir.
- **Interpretabilidad:** los coeficientes sobre variables estandarizadas indican directamente qué variables empujan hacia cada clase lo que lo hace bastante interpretable.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.